In [ ]:
# Input: 7762_Predicted_metabolites.csv with index and predicted metabolite columns.
# Input: 7762_Metadata.xlsx with matching index, Cohort, study_condition, age, gender, and BMI columns; one record per sample.

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from patsy import build_design_matrices
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
from matplotlib.patches import Patch
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


In [ ]:
DATA_PATH = Path("7762_Predicted_metabolites.csv")
META_PATH = Path("7762_Metadata.xlsx")
OUTPUT_DIR = Path("single_cohort_emm_results")

GROUP_COL = "study_condition"
CONTROL_VALUE = "control"
MATCH_COLS = ["Cohort"]

CONTROL_COHORT_OVERRIDES = {"RA": ["QinN_2014"]}

EXCLUDE_CONDITIONS = {"IBD", "CRC", "T2D", "adenoma"}
EXCLUDE_METABOLITES = {"IAM", "IAA", "IA"}

COVARIATE_COLUMNS = ["age", "gender", "BMI"]
MIN_CASE_N = 5
MIN_CONTROL_N = 5
MIN_MODEL_N_PER_GROUP = 5
FDR_METHOD = "fdr_bh"

N_COLS = 4
PANEL_WIDTH = 4.2
PANEL_HEIGHT = 3.5
DPI = 350
HEALTHY_COLOR = "#5DADE2"
CASE_COLOR = "#F06A6A"
DISPLAY_NAME_MAP = {
    "RA": "AS",
    "carcinoma_surgery_history": "CSH",
    "pre-hypertension": "Pre-Hypertension",
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Metabolite data file not found: {DATA_PATH}")
if not META_PATH.exists():
    raise FileNotFoundError(f"Metadata file not found: {META_PATH}")

abundance = pd.read_csv(DATA_PATH)
metadata = pd.read_excel(META_PATH)

required_abundance_cols = {"index"}
required_metadata_cols = {"index", GROUP_COL, *MATCH_COLS}
missing_abundance = required_abundance_cols - set(abundance.columns)
missing_metadata = required_metadata_cols - set(metadata.columns)
if missing_abundance:
    raise KeyError(f"Missing CSV columns: {sorted(missing_abundance)}")
if missing_metadata:
    raise KeyError(f"Missing metadata columns: {sorted(missing_metadata)}")

abundance["index"] = abundance["index"].astype(str).str.strip()
metadata["index"] = metadata["index"].astype(str).str.strip()
if abundance["index"].duplicated().any() or metadata["index"].duplicated().any():
    raise ValueError("Duplicate sample identifiers prevent one-to-one matching.")

candidate_metabolites = [c for c in abundance.columns if c != "index"]
for col in candidate_metabolites:
    abundance[col] = pd.to_numeric(abundance[col], errors="coerce")
METABOLITES = [
    c for c in candidate_metabolites
    if c not in EXCLUDE_METABOLITES and abundance[c].notna().any()
]
if not METABOLITES:
    raise ValueError("No eligible metabolite columns found.")

merged = abundance.merge(metadata, on="index", how="inner", validate="one_to_one")
if len(merged) != len(abundance) or len(merged) != len(metadata):
    raise ValueError(
        f"Sample identifiers do not match completely: CSV={len(abundance)}, metadata={len(metadata)}, merged={len(merged)}"
    )
merged[GROUP_COL] = merged[GROUP_COL].astype(str).str.strip()
merged_raw = merged.copy()

print(f"Matched samples: {len(merged):,}")
print(f"Included metabolites: {METABOLITES}")
display(merged[["index", GROUP_COL, *MATCH_COLS, *METABOLITES]].head())


In [ ]:
def healthy_fit_transform_by_cohort(frame, metabolites):

    transformed = frame.copy()
    fit_rows = []
    if MATCH_COLS:
        grouped = frame.groupby(MATCH_COLS, dropna=False, sort=False)
    else:
        grouped = [("all", frame)]

    for group_key, group in grouped:
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        group_label = " | ".join(str(x) for x in group_key)
        healthy_index = group.index[group[GROUP_COL].eq(CONTROL_VALUE)]
        for metabolite in metabolites:
            healthy_values = frame.loc[healthy_index, metabolite].dropna().to_numpy(dtype=float)
            if len(healthy_values) == 0:
                transformed.loc[group.index, metabolite] = np.nan
                fit_rows.append({
                    "matching_group": group_label,
                    "metabolite": metabolite,
                    "n_healthy_fit": 0,
                    "healthy_mean": np.nan,
                    "healthy_scale": np.nan,
                    "status": "no healthy samples",
                })
                continue

            healthy_mean = float(np.mean(healthy_values))
            healthy_scale = float(np.std(healthy_values, ddof=0))
            if not np.isfinite(healthy_scale) or healthy_scale == 0:
                healthy_scale = 1.0
            transformed.loc[group.index, metabolite] = (
                frame.loc[group.index, metabolite] - healthy_mean
            ) / healthy_scale
            fit_rows.append({
                "matching_group": group_label,
                "metabolite": metabolite,
                "n_healthy_fit": len(healthy_values),
                "healthy_mean": healthy_mean,
                "healthy_scale": healthy_scale,
                "status": "ok",
            })

    return transformed, pd.DataFrame(fit_rows)

merged, transform_fit_summary = healthy_fit_transform_by_cohort(merged_raw, METABOLITES)

external_fit_rows = []
for condition, control_cohorts in CONTROL_COHORT_OVERRIDES.items():
    case_index = merged_raw.index[merged_raw[GROUP_COL].eq(condition)]
    fit_index = merged_raw.index[
        merged_raw[GROUP_COL].eq(CONTROL_VALUE)
        & merged_raw["Cohort"].isin(control_cohorts)
    ]
    if len(case_index) == 0 or len(fit_index) == 0:
        continue
    for metabolite in METABOLITES:
        healthy_values = merged_raw.loc[fit_index, metabolite].dropna().to_numpy(dtype=float)
        if len(healthy_values) == 0:
            merged.loc[case_index, metabolite] = np.nan
            continue
        healthy_mean = float(np.mean(healthy_values))
        healthy_scale = float(np.std(healthy_values, ddof=0))
        if not np.isfinite(healthy_scale) or healthy_scale == 0:
            healthy_scale = 1.0
        merged.loc[case_index, metabolite] = (
            merged_raw.loc[case_index, metabolite] - healthy_mean
        ) / healthy_scale
        external_fit_rows.append({
            "matching_group": f"external control: {', '.join(control_cohorts)}",
            "metabolite": metabolite,
            "n_healthy_fit": len(healthy_values),
            "healthy_mean": healthy_mean,
            "healthy_scale": healthy_scale,
            "status": f"case {condition} transformed with external healthy fit",
        })
if external_fit_rows:
    transform_fit_summary = pd.concat(
        [transform_fit_summary, pd.DataFrame(external_fit_rows)], ignore_index=True
    )

TRANSFORMED_DATA_CSV = OUTPUT_DIR / "metabolite_abundance_healthy_fit_transformed.csv"
merged[["index", *METABOLITES, GROUP_COL, *MATCH_COLS]].to_csv(
    TRANSFORMED_DATA_CSV, index=False, encoding="utf-8-sig"
)
print(f"Standardized data: {TRANSFORMED_DATA_CSV}")
display(transform_fit_summary.head())


In [ ]:
def get_case_and_control(frame, condition):
    case = frame.loc[frame[GROUP_COL].eq(condition)].copy()
    if condition in CONTROL_COHORT_OVERRIDES:
        control = frame.loc[
            frame[GROUP_COL].eq(CONTROL_VALUE)
            & frame["Cohort"].isin(CONTROL_COHORT_OVERRIDES[condition])
        ].copy()
    elif not MATCH_COLS:
        control = frame.loc[frame[GROUP_COL].eq(CONTROL_VALUE)].copy()
    else:
        keys = case[MATCH_COLS].drop_duplicates()
        control = (
            frame.loc[frame[GROUP_COL].eq(CONTROL_VALUE)]
            .merge(keys, on=MATCH_COLS, how="inner")
            .drop_duplicates(subset="index")
        )
        matched_keys = control[MATCH_COLS].drop_duplicates()
        case = case.merge(matched_keys, on=MATCH_COLS, how="inner").drop_duplicates(subset="index")
    return control, case

def available_covariates(control, case):

    return [
        col for col in COVARIATE_COLUMNS
        if case[col].notna().any() and control[col].notna().any()
    ]

def case_has_any_covariate(case):
    return any(case[col].notna().any() for col in COVARIATE_COLUMNS)

candidates = [
    x for x in merged[GROUP_COL].dropna().unique()
    if x != CONTROL_VALUE and x not in EXCLUDE_CONDITIONS
]
panel_rows = []
excluded_rows = []
for condition in candidates:
    raw_case = merged.loc[merged[GROUP_COL].eq(condition)].copy()
    raw_case_cohorts = raw_case["Cohort"].dropna().unique()
    if len(raw_case_cohorts) != 1:
        excluded_rows.append({
            "condition": condition,
            "condition_label": DISPLAY_NAME_MAP.get(condition, condition),
            "reason": "case samples span multiple Cohort values",
            "n_case": len(raw_case),
            "n_healthy": np.nan,
        })
        continue

    control, case = get_case_and_control(merged, condition)
    case_cohorts = raw_case_cohorts
    if not case_has_any_covariate(case):
        excluded_rows.append({
            "condition": condition,
            "condition_label": DISPLAY_NAME_MAP.get(condition, condition),
            "reason": "age, gender and BMI are all NA in the disease group",
            "n_case": len(case),
            "n_healthy": len(control),
        })
        continue
    if len(case) < MIN_CASE_N or len(control) < MIN_CONTROL_N:
        excluded_rows.append({
            "condition": condition,
            "condition_label": DISPLAY_NAME_MAP.get(condition, condition),
            "reason": "insufficient matched samples",
            "n_case": len(case),
            "n_healthy": len(control),
        })
        continue
    panel_rows.append({
        "condition": condition,
        "condition_label": DISPLAY_NAME_MAP.get(condition, condition),
        "n_case": len(case),
        "n_healthy": len(control),
        "case_cohort": str(case_cohorts[0]),
        "adjustable_covariates": ", ".join(available_covariates(control, case)) or "none",
    })

panel_qc = pd.DataFrame(panel_rows)
excluded_conditions = pd.DataFrame(excluded_rows)
PANEL_CONDITIONS = panel_qc.sort_values(
    "condition_label", key=lambda s: s.astype(str).str.casefold()
)["condition"].tolist()

if not PANEL_CONDITIONS:
    raise ValueError("No disease comparisons satisfy the cohort and sample-size criteria.")

print(f"Included disease comparisons: {len(PANEL_CONDITIONS)}")
display(panel_qc)
if not excluded_conditions.empty:
    print("Excluded diseases or conditions: ")
    display(excluded_conditions)


In [ ]:
def _wald_ci(estimate, standard_error, z_value=1.96):
    if not np.isfinite(standard_error):
        return np.nan, np.nan
    return estimate - z_value * standard_error, estimate + z_value * standard_error

def fit_emm_for_metabolite(panel_data, metabolite, covariates):
    model_columns = ["status", metabolite, *covariates]
    model_data = panel_data[model_columns].copy()
    model_data[metabolite] = pd.to_numeric(model_data[metabolite], errors="coerce")
    for col in ["age", "BMI"]:
        if col in model_data.columns:
            model_data[col] = pd.to_numeric(model_data[col], errors="coerce")
    model_data = model_data.dropna(subset=model_columns).copy()
    if "gender" in model_data.columns:

        model_data["gender"] = model_data["gender"].astype(object)
    group_counts = model_data["status"].value_counts()
    if (
        group_counts.get(0, 0) < MIN_MODEL_N_PER_GROUP
        or group_counts.get(1, 0) < MIN_MODEL_N_PER_GROUP
    ):
        return None

    formula_terms = ["status"]
    for covariate in covariates:
        formula_terms.append("C(gender)" if covariate == "gender" else covariate)
    formula = f'Q("{metabolite}") ~ ' + " + ".join(formula_terms)
    model = smf.ols(formula, data=model_data).fit(cov_type="HC3")
    design_info = model.model.data.design_info
    beta = np.asarray(model.params, dtype=float)
    robust_cov = np.asarray(model.cov_params(), dtype=float)

    def mean_design_for_status(status_value):
        counterfactual = model_data.copy()
        counterfactual["status"] = status_value
        design = np.asarray(
            build_design_matrices(
                [design_info], counterfactual, return_type="dataframe"
            )[0],
            dtype=float,
        )
        return design.mean(axis=0)

    l_healthy = mean_design_for_status(0)
    l_disease = mean_design_for_status(1)
    difference_vector = l_disease - l_healthy

    def estimate_and_se(vector):
        estimate = float(vector @ beta)
        variance = float(vector @ robust_cov @ vector)
        standard_error = float(np.sqrt(max(variance, 0.0)))
        return estimate, standard_error

    healthy_estimate, healthy_se = estimate_and_se(l_healthy)
    disease_estimate, disease_se = estimate_and_se(l_disease)
    difference, difference_se = estimate_and_se(difference_vector)
    healthy_low, healthy_high = _wald_ci(healthy_estimate, healthy_se)
    disease_low, disease_high = _wald_ci(disease_estimate, disease_se)
    difference_low, difference_high = _wald_ci(difference, difference_se)
    if difference_se == 0:
        p_value = 0.0 if difference != 0 else 1.0
    else:
        p_value = float(2 * norm.sf(abs(difference / difference_se)))

    return {
        "formula": formula,
        "n_model_healthy": int(group_counts.get(0, 0)),
        "n_model_disease": int(group_counts.get(1, 0)),
        "emm_healthy": healthy_estimate,
        "emm_healthy_ci_low": healthy_low,
        "emm_healthy_ci_high": healthy_high,
        "emm_disease": disease_estimate,
        "emm_disease_ci_low": disease_low,
        "emm_disease_ci_high": disease_high,
        "adjusted_difference_disease_minus_healthy": difference,
        "difference_ci_low": difference_low,
        "difference_ci_high": difference_high,
        "p_value": p_value,
    }

emm_rows = []
for condition in PANEL_CONDITIONS:
    control, case = get_case_and_control(merged, condition)
    control = control.assign(status=0)
    case = case.assign(status=1)
    panel_data = pd.concat([control, case], ignore_index=True)
    covariates = available_covariates(control, case)
    for metabolite in METABOLITES:
        result = fit_emm_for_metabolite(panel_data, metabolite, covariates)
        if result is None:
            continue
        emm_rows.append({
            "condition": condition,
            "condition_label": DISPLAY_NAME_MAP.get(condition, condition),
            "metabolite": metabolite,
            "adjusted_for": ", ".join(covariates) if covariates else "none",
            **result,
        })

summary = pd.DataFrame(emm_rows)
if summary.empty:
    raise ValueError("No metabolite models satisfy the minimum sample-size requirement.")

summary["q_value"] = np.nan
for condition, idx in summary.groupby("condition").groups.items():
    summary.loc[idx, "q_value"] = multipletests(
        summary.loc[idx, "p_value"], method=FDR_METHOD
    )[1]

def significance_stars(q_value):
    if pd.isna(q_value):
        return ""
    if q_value <= 0.0001:
        return "****"
    if q_value <= 0.001:
        return "***"
    if q_value <= 0.01:
        return "**"
    if q_value < 0.1:
        return "*"
    return ""

summary["stars"] = summary["q_value"].map(significance_stars)
summary["significant_q10"] = summary["q_value"].lt(0.1)
summary = summary.sort_values(
    ["q_value", "condition_label", "metabolite"]
).reset_index(drop=True)
significant_summary = summary.loc[summary["significant_q10"]].copy()

print(f"Completed EMM models: {len(summary)}")
print(f"Results with q < 0.1: {len(significant_summary)}")
display(summary)


In [ ]:
def add_bracket(ax, x1, x2, y, height, label):
    ax.plot([x1, x1, x2, x2], [y, y + height, y + height, y],
            lw=0.9, c="black", clip_on=False)
    ax.text((x1 + x2) / 2, y + height, label,
            ha="center", va="bottom", fontsize=9, clip_on=False)

n_rows = int(np.ceil(len(PANEL_CONDITIONS) / N_COLS))
fig, axes = plt.subplots(
    n_rows, N_COLS,
    figsize=(PANEL_WIDTH * N_COLS, PANEL_HEIGHT * n_rows),
    squeeze=False,
)
axes_flat = axes.ravel()

for ax, condition in zip(axes_flat, PANEL_CONDITIONS):
    control, case = get_case_and_control(merged, condition)
    panel_stats = summary.loc[summary["condition"].eq(condition)].copy()
    panel_stats = panel_stats.set_index("metabolite")
    covariate_text = panel_stats["adjusted_for"].iloc[0]

    for i, metabolite in enumerate(METABOLITES):
        if metabolite not in panel_stats.index:
            continue
        stat = panel_stats.loc[metabolite]
        x_healthy = i - 0.18
        x_disease = i + 0.18
        ax.errorbar(
            x_healthy, stat["emm_healthy"],
            yerr=[[stat["emm_healthy"] - stat["emm_healthy_ci_low"]],
                  [stat["emm_healthy_ci_high"] - stat["emm_healthy"]]],
            fmt="o", ms=4.5, color=HEALTHY_COLOR, ecolor="black",
            elinewidth=0.9, capsize=3, zorder=3,
        )
        ax.errorbar(
            x_disease, stat["emm_disease"],
            yerr=[[stat["emm_disease"] - stat["emm_disease_ci_low"]],
                  [stat["emm_disease_ci_high"] - stat["emm_disease"]]],
            fmt="o", ms=4.5, color=CASE_COLOR, ecolor="black",
            elinewidth=0.9, capsize=3, zorder=3,
        )

    plot_values = panel_stats[[
        "emm_healthy_ci_low", "emm_healthy_ci_high",
        "emm_disease_ci_low", "emm_disease_ci_high",
    ]].to_numpy(dtype=float).ravel()
    plot_values = plot_values[np.isfinite(plot_values)]
    if len(plot_values):
        y_min = float(np.min(plot_values))
        y_max = float(np.max(plot_values))
        y_range = max(y_max - y_min, 1e-6)
        sig_stats = panel_stats.loc[panel_stats["q_value"].lt(0.1)]
        annotation_top = y_max + 0.12 * y_range
        for rank, (metabolite, stat) in enumerate(sig_stats.iterrows()):
            i = METABOLITES.index(metabolite)
            local_max = max(stat["emm_healthy_ci_high"], stat["emm_disease_ci_high"])
            annotation_top = max(annotation_top, local_max + (0.08 + 0.10 * rank) * y_range)
        ax.set_ylim(y_min - 0.12 * y_range, annotation_top + 0.08 * y_range)
        for rank, (metabolite, stat) in enumerate(sig_stats.iterrows()):
            i = METABOLITES.index(metabolite)
            local_max = max(stat["emm_healthy_ci_high"], stat["emm_disease_ci_high"])
            y = local_max + (0.08 + 0.10 * rank) * y_range
            add_bracket(ax, i - 0.18, i + 0.18, y, 0.02 * y_range, stat["stars"])

    ax.set_xticks(np.arange(len(METABOLITES)))
    ax.set_xticklabels(METABOLITES, rotation=45, ha="right", fontsize=8)
    ax.set_xlim(-0.65, len(METABOLITES) - 0.35)
    ax.grid(axis="both", color="#D9D9D9", linewidth=0.65)
    ax.set_axisbelow(True)
    ax.tick_params(axis="both", labelsize=8)
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
        spine.set_color("#555555")

    ax.set_title(
        f"{DISPLAY_NAME_MAP.get(condition, condition)}   H/C = {len(control)}/{len(case)}",
        fontsize=10, fontweight="bold", pad=6,
    )
    ax.text(0.98, 0.98, f"adjusted: {covariate_text}",
            transform=ax.transAxes, ha="right", va="top", fontsize=7.5)

for ax in axes_flat[len(PANEL_CONDITIONS):]:
    ax.remove()

fig.supxlabel("Metabolite", fontsize=11, y=0.02)
fig.supylabel("Healthy-fit z-score: adjusted EMM ± 95% CI", fontsize=11, x=0.02)
fig.suptitle("Covariate-adjusted metabolite differences", fontsize=14, y=0.995)
fig.legend(
    handles=[
        Patch(facecolor=HEALTHY_COLOR, edgecolor="black", label="Healthy EMM"),
        Patch(facecolor=CASE_COLOR, edgecolor="black", label="Disease EMM"),
    ],
    loc="upper right", bbox_to_anchor=(0.995, 0.995),
    frameon=False, ncol=2, fontsize=9,
)
fig.tight_layout(rect=[0.03, 0.04, 1, 0.97])

FIG_PNG = OUTPUT_DIR / "different_disease_metabolite_EMM_95CI.png"
FIG_PDF = OUTPUT_DIR / "different_disease_metabolite_EMM_95CI.pdf"
fig.savefig(FIG_PNG, dpi=DPI, bbox_inches="tight", facecolor="white")
fig.savefig(FIG_PDF, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Figure saved: {FIG_PNG}")
print(f"Vector figure saved: {FIG_PDF}")


In [ ]:
ALL_SUMMARY_CSV = OUTPUT_DIR / "metabolite_disease_EMM_all_tests.csv"
SIG_SUMMARY_CSV = OUTPUT_DIR / "metabolite_disease_EMM_significant_q10.csv"
PANEL_QC_CSV = OUTPUT_DIR / "metabolite_disease_EMM_panel_qc.csv"
EXCLUDED_CSV = OUTPUT_DIR / "metabolite_disease_EMM_excluded_conditions.csv"
TRANSFORM_FIT_CSV = OUTPUT_DIR / "healthy_fit_transform_parameters.csv"

summary.to_csv(ALL_SUMMARY_CSV, index=False, encoding="utf-8-sig")
significant_summary.to_csv(SIG_SUMMARY_CSV, index=False, encoding="utf-8-sig")
panel_qc.to_csv(PANEL_QC_CSV, index=False, encoding="utf-8-sig")
excluded_conditions.to_csv(EXCLUDED_CSV, index=False, encoding="utf-8-sig")
transform_fit_summary.to_csv(TRANSFORM_FIT_CSV, index=False, encoding="utf-8-sig")

print(f"All EMM results: {ALL_SUMMARY_CSV}")
print(f"Significant EMM results: {SIG_SUMMARY_CSV}")
print(f"Comparison QC: {PANEL_QC_CSV}")
print(f"Exclusion list: {EXCLUDED_CSV}")
print(f"Control standardization parameters: {TRANSFORM_FIT_CSV}")
